In [10]:
q1: str = "I just discovered the course, can I still join?"
q2: str = "I just found out about the program, can I still enroll?"

In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
v1 = model.encode(q1) # embedding for the first question

In [4]:
v1.shape

(384,)

In [5]:
v2 = model.encode(q2) # embedding for the second question

In [6]:
v2.shape

(384,)

In [7]:
d: str = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(v2) # un cosinus proche de 1 veut dire : « les deux vecteurs pointent presque dans la même direction », ce qui se traduit par « les deux phrases ont presque le même sens

np.float32(0.6227728)

In [9]:
v1.dot(dv)

np.float32(0.39572883)

In [10]:
q3: str = 'How to install Docker on Windows?'
v3 = model.encode(q3)

In [11]:
v2.dot(v3)

np.float32(0.009584978)

In [11]:
from ingest import load_faq_data

documents = load_faq_data()

In [13]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [13]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

len(texts)

1350

In [14]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)): # génère les indices de départ de chaque batch : 0, 50, 100, 150, ...
    batch = texts[i:i + batch_size] # Découpe une tranche de 50 textes : texts[0:50], puis texts[50:100], etc.
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors) # extend ajoute les 50 vecteurs un par un à vectors (contrairement à append qui ajouterait la liste entière comme un seul élément).

  0%|          | 0/27 [00:00<?, ?it/s]

In [16]:
len(vectors)

1350

In [15]:
import numpy as np
X = np.array(vectors)

In [18]:
X.shape

(1350, 384)

In [17]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [18]:
scores = X.dot(v_query)

In [19]:
scores = np.array([v_query.dot(X[i]) for i in range(len(X))])

In [20]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [23]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [21]:
top5 = np.argsort(scores)[-5:]

In [22]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [23]:
scores[top5]

array([0.76294106, 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [30]:
top5 = np.argsort(-scores)[:5]

In [31]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [25]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [34]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [26]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [27]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [28]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [29]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [30]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [31]:
from rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [35]:
vector_assistant.rag("How much time is needed to finish the course ?")

"I don't know."

In [ ]:
# Vector search with sqlitesearch

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", # 10K-500K vectors, K-means clustering
    db_path="faq_vectors2.db"
)

In [37]:
# Index data

vs_index.fit(vectors, documents)

In [38]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [39]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [40]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)